# Ukraine

The footage in social media that is [being reported](https://www.nytimes.com/live/2022/02/25/world/russia-ukraine-war/ukraine-says-russian-troops-entered-the-outskirts-of-kyiv) about the Russian invasion of Ukraine is just astonishing. How can you use the Twitter API to tap into some of this?

To get started you need to install and configure twarc with [Twitter Developer](https://developer.twitter.com/en) keys:

In [ ]:
! pip install twarc
! twarc2 configure

Now lets [build a query](https://developer.twitter.com/en/docs/twitter-api/tweets/search/integrate/build-a-query) to collect some tweets:

* tweets mentioning *Ukraine*
* no retweets (cuts down on the number of tweets that need to be retrieved)
* tweets containing videos
* tweets since midnight yesterday

In [ ]:
! twarc2 search --start-time '2022-02-24T00:00:00' --end-time '2022-02-25T21:25:00' 'ukraine has:videos -is:retweet' data/ukraine.jsonl 

Now lets install the twarc-csv module to convert them to CSV to make them easier to process:

In [ ]:
! pip install twarc-csv

Now we can convert them:

In [ ]:
! twarc2 csv data/ukraine.jsonl data/ukraine.csv

Now the data can be loaded into a Pandas DataFrame:

In [ ]:
import pandas

df = pandas.read_csv('data/ukraine.csv')
df.head()

There are a lot of columns in this DataFrame:

In [ ]:
df.columns

Lets sort the DataFrame in descending order by the number of times it was retweeted:

In [ ]:
df = df.sort_values('public_metrics.retweet_count', ascending=False)

Lets see what these tweets are:

In [ ]:
for i, tweet in df.head(50).iterrows():
    print(f"https://twitter.com/{tweet['author.username']}/status/{tweet['id']}")

These don't appear at first blush to be footage. Lets limit to tweets that are in Ukrainian and sort those.

In [ ]:
uk_tweets = df[df['lang'] == 'uk']
uk_tweets

In [ ]:
uk_tweets = uk_tweets.sort_values('public_metrics.retweet_count', ascending=False)

for i, tweet in uk_tweets.iterrows():
    print(f"https://twitter.com/{tweet['author.username']}/status/{tweet['id']}")

This seemed to work better. We could search again and limit to tweets that mention Russia or Russian in Ukrainian:

* ?????????? 
* ?????

And lets increase the time frame to get one more day.


In [ ]:
! twarc2 search --start-time '2022-02-23T00:00:00' --end-time '2022-02-25T21:25:00' '(?????????? OR ?????) has:videos -is:retweet' data/ukrainian.jsonl 
! twarc2 csv data/ukrainian.jsonl data/ukrainian.csv

In [ ]:
df2 = pandas.read_csv('data/ukrainian.csv')

uk_tweets = uk_tweets.sort_values('public_metrics.retweet_count', ascending=False)
for i, tweet in uk_tweets.iterrows():
    print(f"https://twitter.com/{tweet['author.username']}/status/{tweet['id']}")

We can use the twarc-videos plugin to collect these.

In [ ]:
!pip install twarc-videos

In [ ]:
! twarc2 videos --help

In [ ]:
! twarc2 videos data/ukrainian.jsonl --download-dir ~/ukrainian-videos